# 10_heart_liana_summary

Selected source cells for review and reconstruction.

Source: `Heart_combined_samples/CellCellInteraction_Aging_1/Python LIANA+ Heart Overall.ipynb`

All four Core/Neighbor/Distal exports. Figure styling trials excluded.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


Source cell 2 (one-based); original execution count 1.

In [ ]:
import os
import pandas as pd
import scanpy as sc
import scipy.io as sio
import liana as li

def check_export(outdir: str):
    req = ["X.mtx", "genes.txt", "cells.txt", "obs.csv"]
    missing = [f for f in req if not os.path.exists(os.path.join(outdir, f))]
    if missing:
        raise FileNotFoundError(f"{outdir}: missing {missing}")
    return True

def load_export(outdir: str) -> sc.AnnData:
    check_export(outdir)
    X = sio.mmread(f"{outdir}/X.mtx").tocsr().T.tocsr()  # cells x genes
    genes = pd.read_csv(f"{outdir}/genes.txt", header=None)[0].tolist()
    cells = pd.read_csv(f"{outdir}/cells.txt", header=None)[0].tolist()
    obs = pd.read_csv(f"{outdir}/obs.csv").set_index("cell_id").loc[cells]

    adata = sc.AnnData(X=X)
    adata.var_names = genes
    adata.obs_names = cells
    adata.obs = obs
    return adata

# ---- 0) outdir確認 ----
exports = {
    "Heart_Aged1":  "Aged1_LIANA_export",
    "Heart_Aged2":  "Aged2_LIANA_export",
    "Heart_Young1": "Young1_LIANA_export",
    "Heart_Young2": "Young2_LIANA_export",
}
for tag, outdir in exports.items():
    check_export(outdir)
print("OK: all exports exist")

Source cell 3 (one-based); original execution count 2.

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import scipy.io as sio
import liana as li
import matplotlib.pyplot as plt

def load_export(outdir: str) -> sc.AnnData:
    X = sio.mmread(f"{outdir}/X.mtx").tocsr().T.tocsr()  # cells x genes
    genes = pd.read_csv(f"{outdir}/genes.txt", header=None)[0].tolist()
    cells = pd.read_csv(f"{outdir}/cells.txt", header=None)[0].tolist()
    obs = pd.read_csv(f"{outdir}/obs.csv").set_index("cell_id").loc[cells]
    adata = sc.AnnData(X=X)
    adata.var_names = genes
    adata.obs_names = cells
    adata.obs = obs
    return adata

def neighbor_distal_keep(res_df: pd.DataFrame):
    df = res_df.copy()
    df["lr"] = df["ligand_complex"].astype(str) + "→" + df["receptor_complex"].astype(str)

    nb = df[(df["target"]=="Core") & (df["source"].str.startswith("Neighbor_"))].copy()
    ds = df[(df["target"]=="Core") & (df["source"].str.startswith("Distal_"))].copy()

    nb = nb[~nb["source"].isin({"Neighbor_Erythroid cell","Neighbor_Unknown"})].copy()
    ds = ds[~ds["source"].isin({"Distal_Erythroid cell","Distal_Unknown"})].copy()

    nb["ct"] = nb["source"].str.replace("^Neighbor_", "", regex=True)
    ds["ct"] = ds["source"].str.replace("^Distal_", "", regex=True)

    nb_mean = nb.groupby(["ct","lr"], as_index=False)["lrscore"].mean().rename(columns={"lrscore":"mean_neighbor"})
    ds_mean = ds.groupby(["ct","lr"], as_index=False)["lrscore"].mean().rename(columns={"lrscore":"mean_distal"})

    m = nb_mean.merge(ds_mean, on=["ct","lr"], how="left")
    m["mean_distal"] = m["mean_distal"].fillna(0.0)
    m["delta"] = m["mean_neighbor"] - m["mean_distal"]

    m_keep = m[m["delta"] > 0].copy()
    return df, m, m_keep

def run_one(tag: str, outdir: str):
    # load + LIANA
    adata = load_export(outdir)
    li.mt.rank_aggregate(
        adata,
        groupby="CCI_group",
        resource_name="mouseconsensus",
        use_raw=False,
        verbose=True
    )
    res = adata.uns["liana_res"].copy()

    # keep
    df, m, keep = neighbor_distal_keep(res)

    # dotplot
    dfp = keep.copy()
    if dfp.shape[0] == 0:
        print(f"[{tag}] keep=0 (no Neighbor>Distal)")
        return {"tag": tag, "n_cells": adata.n_obs, "n_genes": adata.n_vars, "m": m.shape, "keep": keep.shape}

    s = dfp["mean_neighbor"].astype(float).values
    dfp["size"] = (np.sqrt(s) - np.sqrt(s.min()) + 1e-6) * 250

    ct_order = (dfp.groupby("ct")["mean_neighbor"].mean()
                .sort_values(ascending=False).index.tolist())
    lr_order = (dfp.groupby("lr")["mean_neighbor"].max()
                .sort_values(ascending=False).index.tolist())

    dfp["ct"] = pd.Categorical(dfp["ct"], categories=ct_order, ordered=True)
    dfp["lr"] = pd.Categorical(dfp["lr"], categories=lr_order, ordered=True)

    plt.figure(figsize=(3.3, max(6, 0.22*len(lr_order))))
    plt.scatter(x=dfp["ct"], y=dfp["lr"], s=dfp["size"], alpha=0.7)
    plt.xticks(rotation=45, ha="right")
    plt.title(f"{tag}: Neighbor>Distal LR (size=mean lrscore Neighbor→Core)")
    plt.xlabel("")
    plt.ylabel("Ligand→Receptor")
    plt.tight_layout()
    plt.show()

    return {"tag": tag, "n_cells": adata.n_obs, "n_genes": adata.n_vars, "m": m.shape, "keep": keep.shape}

# ---- 4サンプル回す ----
exports = {
    "Heart_Aged1":  "Aged1_LIANA_export",
    "Heart_Aged2":  "Aged2_LIANA_export",
    "Heart_Young1": "Young1_LIANA_export",
    "Heart_Young2": "Young2_LIANA_export",
}

summ = []
for tag, outdir in exports.items():
    print(f"=== {tag} ({outdir}) ===")
    summ.append(run_one(tag, outdir))

pd.DataFrame(summ)

Source cell 5 (one-based); original execution count 4.

In [ ]:
def run_one_keep(tag: str, outdir: str):
    adata = load_export(outdir)
    li.mt.rank_aggregate(
        adata,
        groupby="CCI_group",
        resource_name="mouseconsensus",
        use_raw=False,
        verbose=True
    )
    res = adata.uns["liana_res"].copy()
    _, m, keep = neighbor_distal_keep(res)
    keep["tag"] = tag
    return keep

keeps = []
for tag, outdir in exports.items():
    print(f"=== {tag} ({outdir}) ===")
    keeps.append(run_one_keep(tag, outdir))

all_keep = pd.concat(keeps, ignore_index=True)
all_keep.head()

Source cell 6 (one-based); original execution count 6.

In [ ]:
for tag, df in all_keep.groupby("tag"):
    df.drop(columns=["tag"]).to_csv(f"{tag}_keep.csv", index=False)

Source cell 7 (one-based); original execution count 7.

In [ ]:
import pandas as pd
import numpy as np

def combine_two_from_all(k: pd.DataFrame, tags: list[str], prefix: str) -> pd.DataFrame:
    k1 = k[k["tag"] == tags[0]][["ct","lr","mean_neighbor"]].rename(columns={"mean_neighbor": f"{prefix}1"})
    k2 = k[k["tag"] == tags[1]][["ct","lr","mean_neighbor"]].rename(columns={"mean_neighbor": f"{prefix}2"})
    m = k1.merge(k2, on=["ct","lr"], how="outer")
    m[f"{prefix}_present_n"] = (~m[f"{prefix}1"].isna()).astype(int) + (~m[f"{prefix}2"].isna()).astype(int)
    m[f"{prefix}_mean"] = m[[f"{prefix}1", f"{prefix}2"]].mean(axis=1, skipna=True)
    return m

aged_comb = combine_two_from_all(all_keep, ["Heart_Aged1","Heart_Aged2"], "Aged")
young_comb = combine_two_from_all(all_keep, ["Heart_Young1","Heart_Young2"], "Young")

both = aged_comb.merge(young_comb, on=["ct","lr"], how="outer")
both.head(), both.shape

Source cell 8 (one-based); original execution count 9.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- long形式を作る（plot_longを必ず定義）---
aged_long = aged_comb[["ct","lr","Aged_mean","Aged_present_n"]].copy()
aged_long["col"] = "Aged_" + aged_long["ct"]
aged_long = aged_long.rename(columns={"Aged_mean":"value", "Aged_present_n":"present_n"})

young_long = young_comb[["ct","lr","Young_mean","Young_present_n"]].copy()
young_long["col"] = "Young_" + young_long["ct"]
young_long = young_long.rename(columns={"Young_mean":"value", "Young_present_n":"present_n"})

plot_long = pd.concat([aged_long, young_long], ignore_index=True)
plot_long = plot_long.dropna(subset=["value"]).copy()

# --- 並び順 ---
col_order = (plot_long.groupby("col")["value"].mean()
             .sort_values(ascending=False).index.tolist())
lr_order = (plot_long.groupby("lr")["value"].max()
            .sort_values(ascending=False).index.tolist())

dfp = plot_long.copy()
dfp["col"] = pd.Categorical(dfp["col"], categories=col_order, ordered=True)
dfp["lr"]  = pd.Categorical(dfp["lr"],  categories=lr_order, ordered=True)

# --- dot size ---
s = dfp["value"].astype(float).values
dfp["size"] = (np.sqrt(s) - np.sqrt(s.min()) + 1e-6) * 250

plt.figure(figsize=(max(6, 0.22*len(col_order)), max(6, 0.22*len(lr_order))))
plt.scatter(x=dfp["col"], y=dfp["lr"], s=dfp["size"], alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.xlabel("")
plt.ylabel("Ligand→Receptor")
plt.title("Heart: Aged/Young mean lrscore (Neighbor→Core, delta>0 kept)")
plt.tight_layout()
plt.show()

Source cell 16 (one-based); original execution count 38.

In [ ]:
import pandas as pd

# 1) Aged/Youngを (ct, lr) で結合（outer）
m = aged_comb[["ct","lr","Aged_mean","Aged_present_n"]].merge(
    young_comb[["ct","lr","Young_mean","Young_present_n"]],
    on=["ct","lr"], how="outer"
)

# 2) ligand/receptorに分解（lrは "Ligand→Receptor"）
m["ligand_complex"] = m["lr"].str.split("→", n=1).str[0]
m["receptor_complex"] = m["lr"].str.split("→", n=1).str[1]

# 3) 列順を整える
m_out = m[[
    "ct",
    "ligand_complex",
    "receptor_complex",
    "lr",
    "Aged_mean","Aged_present_n",
    "Young_mean","Young_present_n"
]].sort_values(["ct","ligand_complex","receptor_complex"])

# 4) 保存（Heart版）
m_out.to_csv("Heart_Neighbor_to_Core_LR_table_AgedYoung.csv", index=False)

m_out.head()